# 04 — EDAT: Embedding-space Data Augmentation Training

**Prerequisites**:
- `00_csi_pipeline.ipynb` — UNIFIED.jsonl + model definition
- `01_tokenization.ipynb` — token cache in `datasets/token_cache/`
- `02_training.ipynb` — base training
- `03_advanced_training(SCL_&_R_Drop).ipynb` — latest checkpoint (saved as `latest.pt`)

### What this notebook does
1. **[22a] PGD perturbation loop** — gradient ASCENT in embedding space to find worst-case input
2. **[23a] Adversarial KL loss** — forces model to produce consistent predictions on clean vs perturbed input
3. **[24a] Small batch test** — full sanity check: shapes, loss values, gradients, optimizer step

### Architecture used (from `03_advanced_training`)
| Property | Value |
|---|---|
| Pooling | CLS token `[:, 0, :]` |
| Head | `nn.Sequential`: LayerNorm -> Linear(768->384) -> GELU -> Linear(384->8) |
| Loss | Weighted CrossEntropyLoss (no focal) |
| Checkpoint key | `model_state` |
| LoRA targets | query, key, value |


## 0 — Install Dependencies

In [1]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == 'Darwin' and platform.machine() == 'arm64'
IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in __import__('os').environ

print(f'Platform      : {platform.system()} {platform.machine()}')
print(f'Apple Silicon : {IS_APPLE_SILICON}')
print(f'Colab         : {IS_COLAB}')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'peft>=0.10', 'torch', 'pyyaml', 'scikit-learn', 'tqdm',
], check=False)
print('Done.')


Platform      : Linux x86_64
Apple Silicon : False
Colab         : True
Done.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1 — Paths, Config & Device

Handles both Colab (Drive) and local Jupyter automatically.

**Drive path note:** `03_advanced_training` uses `dataMiningProject/CSI_Project/` while `02_training` uses `CSI_Project/`. The code below tries both, so it works regardless.

In [5]:
import os, json, platform, sys
from pathlib import Path
import yaml
import torch
import random
import numpy as np

IS_APPLE_SILICON = platform.system() == 'Darwin' and platform.machine() == 'arm64'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # 03_advanced_training uses dataMiningProject path
    BASE_DIR = Path('/content/drive/MyDrive/dataMiningProject/CSI_Project')
    if not (BASE_DIR / 'config.yaml').exists():
        # Fall back to 02_training path
        BASE_DIR = Path('/content/drive/MyDrive/CSI_Project')
    IS_COLAB = True
    print(f'Colab -- Drive mounted, BASE_DIR: {BASE_DIR}')
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath('__file__')))
    if not (BASE_DIR / 'datasets').exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f'Local -- BASE_DIR: {BASE_DIR}')

cfg_path = BASE_DIR / 'config.yaml'
assert cfg_path.exists(), f'Missing config.yaml at {cfg_path}'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

TOKEN_CACHE_DIR = BASE_DIR / cfg['token_cache_dir']
CHECKPOINT_DIR  = BASE_DIR / cfg['checkpoint_dir']
LOG_DIR         = BASE_DIR / cfg['log_dir']
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = cfg['max_length']
CACHE_FILE = TOKEN_CACHE_DIR / f'tokens_maxlen{MAX_LENGTH}.pt'

# Checkpoint: prefer 03_advanced_training output (latest.pt), fall back to 02 (best_model.pt)
CKPT_OPTIONS = [
    (CHECKPOINT_DIR / 'latest.pt',     'model_state'),
    (CHECKPOINT_DIR / 'best_model.pt', 'model_state_dict'),
]

assert CACHE_FILE.exists(), f'Missing cache: {CACHE_FILE} -- run 01_tokenization.ipynb first'

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif IS_APPLE_SILICON and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

SEED = cfg['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'Device     : {DEVICE}')
print(f'CACHE_FILE : {CACHE_FILE}')
print(f'CKPT_DIR   : {CHECKPOINT_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab -- Drive mounted, BASE_DIR: /content/drive/MyDrive/dataMiningProject/CSI_Project
Device     : cuda
CACHE_FILE : /content/drive/MyDrive/dataMiningProject/CSI_Project/datasets/token_cache/tokens_maxlen512.pt
CKPT_DIR   : /content/drive/MyDrive/dataMiningProject/CSI_Project/checkpoints


## 2 — Model Architecture + Load Checkpoint

The model is copied **exactly** from `03_advanced_training(SCL_&_R_Drop).ipynb`.

Two additions for EDAT:
- `forward_embeds()` method: accepts pre-computed embeddings instead of token IDs (needed by PGD)
- Checkpoint loader that tries both `latest.pt` (from 03) and `best_model.pt` (from 02)

**Do not change the architecture here** -- it must match whichever checkpoint you load.

In [6]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from peft import LoraConfig, TaskType, get_peft_model

CWE_8_CLASSES = ['CWE-077', 'CWE-601', 'CWE-022', 'CWE-094',
                 'CWE-089', 'CWE-352', 'CWE-079', 'unknown']
CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


class GraphCodeBERTLoRACWEModel(nn.Module):
    """
    Exact copy of the model from 03_advanced_training(SCL_&_R_Drop).ipynb.

    Key properties:
      - Pooling  : CLS token only  (enc.last_hidden_state[:, 0, :])
      - Head     : nn.Sequential with NO Dropout
      - Buffer   : self.weights  (NOT self.class_weights)
      - forward(): returns {logits, features}; loss only if cwe_labels passed

    Added for EDAT:
      - forward_embeds(): same as forward() but accepts inputs_embeds instead of input_ids
    """
    def __init__(self, model_name='microsoft/graphcodebert-base',
                 num_cwe_classes=8, class_weights=None):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=16, lora_alpha=32,
            target_modules=['query', 'key', 'value'],
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        hidden = self.encoder.config.hidden_size  # 768
        # Head: LayerNorm -> Linear(768->384) -> GELU -> Linear(384->8)
        # No Dropout -- matches 03_advanced_training exactly
        self.cwe_head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, num_cwe_classes),
        )
        # Buffer name is 'weights' -- matches 03_advanced_training
        self.register_buffer(
            'weights',
            class_weights if class_weights is not None else torch.ones(num_cwe_classes),
        )

    def forward(self, input_ids, attention_mask, cwe_labels=None):
        """Standard forward using token IDs."""
        enc      = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                                output_hidden_states=True)
        features = enc.last_hidden_state[:, 0, :]  # CLS token: (B, 768)
        logits   = self.cwe_head(features)
        result   = {'logits': logits, 'features': features}
        if cwe_labels is not None:
            result['loss'] = F.cross_entropy(logits, cwe_labels, weight=self.weights)
        return result

    def forward_embeds(self, inputs_embeds, attention_mask, cwe_labels=None):
        """
        EDAT-only method: accepts pre-computed embedding tensors instead of input_ids.

        Used by pgd_attack(): after computing the perturbed embeddings (clean + delta),
        we skip the embedding lookup and feed the perturbed tensor directly into
        the encoder. Everything after the embedding layer runs identically.

        Args:
            inputs_embeds : (B, L, 768) -- perturbed embedding tensor
            attention_mask: (B, L)       -- unchanged from original batch
            cwe_labels    : (B,) or None
        """
        enc      = self.encoder(inputs_embeds=inputs_embeds,
                                attention_mask=attention_mask,
                                output_hidden_states=True)
        features = enc.last_hidden_state[:, 0, :]  # still CLS -- same pooling
        logits   = self.cwe_head(features)
        result   = {'logits': logits, 'features': features}
        if cwe_labels is not None:
            result['loss'] = F.cross_entropy(logits, cwe_labels, weight=self.weights)
        return result


print('GraphCodeBERTLoRACWEModel defined')
print('  Pooling  : CLS token [:, 0, :]')
print('  Head     : LayerNorm->Linear(768->384)->GELU->Linear(384->8)')
print('  Buffer   : self.weights')
print('  Extra    : forward_embeds() for EDAT')


GraphCodeBERTLoRACWEModel defined
  Pooling  : CLS token [:, 0, :]
  Head     : LayerNorm->Linear(768->384)->GELU->Linear(384->8)
  Buffer   : self.weights
  Extra    : forward_embeds() for EDAT


In [7]:
model = GraphCodeBERTLoRACWEModel(
    model_name      = cfg['model_name'],
    num_cwe_classes = cfg['num_cwe_classes'],
).to(DEVICE)

# Try loading checkpoints in priority order
loaded = False
for ckpt_path, state_key in CKPT_OPTIONS:
    if ckpt_path.exists():
        print(f'Found: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        print(f'  Keys in checkpoint: {list(ckpt.keys())}')
        # Try the expected key first, then the other one
        for key in [state_key, 'model_state', 'model_state_dict']:
            if key in ckpt:
                model.load_state_dict(ckpt[key])
                epoch = ckpt.get('epoch', 'unknown')
                print(f'  Loaded weights from key={key!r}, epoch={epoch}')
                loaded = True
                break
        if loaded:
            break

if not loaded:
    print('No checkpoint found -- model has random weights.')
    print('This is fine for smoke-testing tasks 22a/23a/24a.')
    print('Run 03_advanced_training first for real EDAT training.')

# Verify the embedding layer path (needed by PGD)
# LoRA wraps: model.encoder (PeftModel)
#              -> .base_model (LoraModel)
#              -> .model (RobertaModel)
#              -> .embeddings.word_embeddings (nn.Embedding: 50265 x 768)
embed_layer = model.encoder.base_model.model.embeddings.word_embeddings
print(f'Embed layer shape: {embed_layer.weight.shape}')  # expect (50265, 768)
print(f'Model on device  : {DEVICE}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Found: /content/drive/MyDrive/dataMiningProject/CSI_Project/checkpoints/latest.pt
  Keys in checkpoint: ['epoch', 'model_state', 'optimizer_state', 'scheduler_state', 'best_val_macro_f1', 'loss_history']
  Loaded weights from key='model_state', epoch=20
Embed layer shape: torch.Size([50265, 768])
Model on device  : cuda


## 3 — Load Token Cache + Build Small Batch

We load the pre-tokenized cache from `01_tokenization.ipynb` and pull 8 training samples (one per CWE class) to use as our sanity-check batch in tasks 22a/23a/24a.

We also recompute `CLASS_WEIGHTS` from the training split and set them on the model's `weights` buffer (same formula as `03_advanced_training`).

In [8]:
from torch.utils.data import Dataset, DataLoader

class VulnerabilityDataset(Dataset):
    """Same as 02_training.ipynb and 03_advanced_training."""
    def __init__(self, cache, split):
        if split == 'all':
            indices = list(range(len(cache['split_origins'])))
        else:
            indices = [i for i, s in enumerate(cache['split_origins']) if s == split]
        self.input_ids      = cache['input_ids'][indices]
        self.attention_mask = cache['attention_mask'][indices]
        self.cwe_labels     = cache['cwe_labels'][indices]
        self.binary_labels  = cache['binary_labels'][indices]
        self.global_ids     = cache['global_ids'][indices]

    def __len__(self): return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'cwe_label':      self.cwe_labels[idx],
            'binary_label':   self.binary_labels[idx],
            'global_id':      self.global_ids[idx],
        }

cache    = torch.load(CACHE_FILE, weights_only=True)
train_ds = VulnerabilityDataset(cache, split='train')
print(f'Cache records : {cache["num_records"]:,}')
print(f'Train samples : {len(train_ds):,}')
label_arr = train_ds.cwe_labels.numpy()
print(f'Label counts  : {np.bincount(label_arr, minlength=8)}')
print(f'Label names   : {CWE_8_CLASSES}')

# Recompute class weights (same formula as 03_advanced_training)
counts = np.bincount(label_arr, minlength=8).astype(float)
CLASS_WEIGHTS = torch.tensor(len(train_ds) / (8 * counts), dtype=torch.float).to(DEVICE)
model.weights = CLASS_WEIGHTS  # update the buffer
print(f'CLASS_WEIGHTS : {CLASS_WEIGHTS.cpu().numpy().round(3)}')

# Build a balanced mini-batch: 1 sample per class
SMALL_BATCH_SIZE = 8
per_class = {i: [] for i in range(8)}
for idx in range(len(train_ds)):
    lbl = train_ds.cwe_labels[idx].item()
    if len(per_class[lbl]) == 0:
        per_class[lbl].append(idx)
    if sum(len(v) for v in per_class.values()) >= SMALL_BATCH_SIZE:
        break

sel = []
for v in per_class.values():
    sel.extend(v)
sel = sel[:SMALL_BATCH_SIZE]

sb = [train_ds[i] for i in sel]
INPUT_IDS  = torch.stack([b['input_ids']      for b in sb]).to(DEVICE)
ATTN_MASK  = torch.stack([b['attention_mask'] for b in sb]).to(DEVICE)
CWE_LABELS = torch.stack([b['cwe_label']      for b in sb]).to(DEVICE)

print(f'Small batch shapes:')
print(f'  input_ids      {tuple(INPUT_IDS.shape)}  dtype={INPUT_IDS.dtype}')
print(f'  attention_mask {tuple(ATTN_MASK.shape)}  dtype={ATTN_MASK.dtype}')
print(f'  cwe_labels     {tuple(CWE_LABELS.shape)}  dtype={CWE_LABELS.dtype}')
print(f'  class mix      {[INDEX_TO_CWE[l.item()] for l in CWE_LABELS]}')


Cache records : 14,522
Train samples : 12,994
Label counts  : [ 969 1071 1385  733 2831 1332  918 3755]
Label names   : ['CWE-077', 'CWE-601', 'CWE-022', 'CWE-094', 'CWE-089', 'CWE-352', 'CWE-079', 'unknown']
CLASS_WEIGHTS : [1.676 1.517 1.173 2.216 0.574 1.219 1.769 0.433]
Small batch shapes:
  input_ids      (8, 512)  dtype=torch.int64
  attention_mask (8, 512)  dtype=torch.int64
  cwe_labels     (8,)  dtype=torch.int64
  class mix      ['CWE-077', 'CWE-601', 'CWE-022', 'CWE-094', 'CWE-089', 'CWE-352', 'CWE-079', 'unknown']


---
## Task 22a — PGD Perturbation Loop

**Goal:** Given a batch, find a small change `delta` to the token embeddings that **maximises** the model's loss (makes it as wrong as possible). This is the adversarial attack.

**Why embeddings?** `input_ids` are integers -- gradients can't flow through them. Token embeddings are floating-point vectors in continuous space -- we CAN compute `d(loss)/d(embeddings)` and use it to find the worst-case perturbation.

**PGD algorithm:**
1. Look up clean embeddings from the word embedding table
2. Initialise `delta` as a random tensor in `[-epsilon, epsilon]`
3. Repeat `num_steps` times:
   - Forward pass with `(clean_embeds + delta)` -> compute loss
   - Compute `grad = d(loss)/d(delta)`
   - **Ascent step**: `delta = delta + alpha * sign(grad)` (maximise loss)
   - **Project**: clamp `delta` back into `[-epsilon, epsilon]` ball
4. Return `clean_embeds + final_delta`

**Hyperparameters:**
- `epsilon=0.01` -- max allowed perturbation (controls attack strength)
- `alpha=0.005`  -- step size per iteration (must be < epsilon)
- `num_steps=3`  -- 3 steps balances quality and speed during training

In [16]:
# EDAT hyperparameters -- add these to config.yaml after verifying
EDAT_EPSILON   = 0.001
EDAT_ALPHA     = 0.0005
EDAT_NUM_STEPS = 3      # gradient ascent steps per batch


def pgd_attack(model, input_ids, attention_mask, cwe_labels,
               epsilon=EDAT_EPSILON, alpha=EDAT_ALPHA, num_steps=EDAT_NUM_STEPS):
    """
    PGD adversarial attack in token embedding space.

    Args:
        model          : GraphCodeBERTLoRACWEModel  (must be in train() mode)
        input_ids      : (B, L) long   -- token IDs from tokenizer
        attention_mask : (B, L) int    -- 1=real token, 0=padding
        cwe_labels     : (B,) long     -- ground truth CWE indices
        epsilon        : float         -- max perturbation norm
        alpha          : float         -- step size per iteration
        num_steps      : int           -- number of ascent steps

    Returns:
        perturbed_embeds: (B, L, 768) float tensor, fully detached.
                          Pass to model.forward_embeds() for the KL loss.
    """
    # Step 1: get clean embeddings
    # Path: model.encoder (PeftModel)
    #         -> .base_model (LoraModel)
    #         -> .model (RobertaModel)
    #         -> .embeddings.word_embeddings (nn.Embedding)
    embed_fn    = model.encoder.base_model.model.embeddings.word_embeddings
    embeds_init = embed_fn(input_ids).detach()  # (B, L, 768) -- no grad on weights

    # Step 2: random initialisation inside the epsilon ball
    delta = torch.zeros_like(embeds_init).uniform_(-epsilon, epsilon)
    delta.requires_grad_(True)

    # Step 3: PGD loop
    for _ in range(num_steps):
        out  = model.forward_embeds(embeds_init + delta, attention_mask, cwe_labels)
        loss = out['loss']     # we want to MAXIMISE this
        loss.backward()

        # Ascent: step in direction that increases loss
        with torch.no_grad():
            delta_new = delta + alpha * delta.grad.sign()
            delta_new = torch.clamp(delta_new, -epsilon, epsilon)  # project

        delta = delta_new.detach().requires_grad_(True)

    # Return perturbed embeddings -- fully detached, no computation graph
    return (embeds_init + delta.detach()).detach()


print('pgd_attack() defined')
print(f'  epsilon   = {EDAT_EPSILON}')
print(f'  alpha     = {EDAT_ALPHA}')
print(f'  num_steps = {EDAT_NUM_STEPS}')


pgd_attack() defined
  epsilon   = 0.001
  alpha     = 0.0005
  num_steps = 3


### Smoke-test for pgd_attack()

Verify: output shape is `(B, L, 768)`, output differs from clean embeddings, and max absolute difference is within epsilon.

In [17]:
model.train()  # dropout must be on during PGD

embed_fn     = model.encoder.base_model.model.embeddings.word_embeddings
clean_embeds = embed_fn(INPUT_IDS).detach()

perturbed = pgd_attack(model, INPUT_IDS, ATTN_MASK, CWE_LABELS)

diff = (perturbed - clean_embeds).abs()
max_diff  = diff.max().item()
mean_diff = diff.mean().item()

assert perturbed.shape == clean_embeds.shape, \
    f'Shape mismatch: {perturbed.shape} vs {clean_embeds.shape}'
assert (perturbed != clean_embeds).any(), \
    'ERROR: perturbed == clean -- PGD did nothing'
assert max_diff <= EDAT_EPSILON + 1e-6, \
    f'ERROR: max delta {max_diff:.6f} > epsilon {EDAT_EPSILON} -- projection failed'

print('PGD smoke-test PASSED')
print(f'  Output shape   : {tuple(perturbed.shape)}  (B, L, 768)')
print(f'  Max  |delta|   : {max_diff:.6f}  (must be <= {EDAT_EPSILON})')
print(f'  Mean |delta|   : {mean_diff:.6f}  (non-zero = perturbation applied)')


PGD smoke-test PASSED
  Output shape   : (8, 512, 768)  (B, L, 768)
  Max  |delta|   : 0.001000  (must be <= 0.001)
  Mean |delta|   : 0.000628  (non-zero = perturbation applied)


---
## Task 23a — Adversarial KL Loss

**Goal:** Combine two signals into one training loss:
1. **CE loss on clean input** -- standard classification, same as `03_advanced_training`
2. **KL divergence** -- forces the model to give the same output distribution on perturbed input as on clean input

**Formula:** `total = CE(clean) + 0.5 * KL(p_clean || p_adv)`

**What is KL divergence here?**
The model outputs a probability distribution over 8 CWE classes. KL divergence measures how different the adversarial distribution is from the clean one. By minimising it, we force the model to be robust: 'even if I perturb the embeddings, my prediction should not change.'

**Critical PyTorch detail:**
`F.kl_div(input, target)` requires:
- `input`  = **log**-probabilities -> use `F.log_softmax`
- `target` = probabilities -> use `F.softmax`
Swapping these gives wrong values with no error message.

In [18]:
EDAT_LAMBDA_ADV = 0.5  # weight of adversarial KL term


def adversarial_kl_loss(model, input_ids, attention_mask, cwe_labels,
                        epsilon=EDAT_EPSILON, alpha=EDAT_ALPHA,
                        num_steps=EDAT_NUM_STEPS, lambda_adv=EDAT_LAMBDA_ADV):
    """
    EDAT adversarial training loss.
    Formula: total = CE_weighted(clean) + lambda_adv * KL(p_clean || p_adv)

    Loss type: weighted CrossEntropy -- matches 03_advanced_training.

    Returns:
        total_loss : scalar tensor (has gradient graph -- call .backward() on this)
        ce_val     : float (clean CE value, for logging)
        kl_val     : float (KL value, for logging)
    """
    # A: Clean forward pass
    clean_out    = model(input_ids, attention_mask, cwe_labels=cwe_labels)
    ce_loss      = clean_out['loss']    # weighted CE on clean input
    clean_logits = clean_out['logits']  # (B, 8)

    # Detach clean probs -- they are the TARGET for KL, not the trained output
    p_clean = torch.softmax(clean_logits.detach(), dim=-1)  # (B, 8)

    # B: PGD attack -- find worst-case perturbed embeddings
    perturbed_embeds = pgd_attack(
        model, input_ids, attention_mask, cwe_labels,
        epsilon=epsilon, alpha=alpha, num_steps=num_steps,
    )  # (B, L, 768) detached

    # C: Adversarial forward pass
    adv_out    = model.forward_embeds(perturbed_embeds, attention_mask)
    adv_logits = adv_out['logits']                          # (B, 8)
    log_p_adv  = F.log_softmax(adv_logits, dim=-1)         # (B, 8) log-probs

    # D: KL divergence
    # F.kl_div(log_input, target) -- log_input=adv, target=clean
    # batchmean: sum over classes, divide by batch size
    kl_loss = F.kl_div(input=log_p_adv, target=p_clean, reduction='batchmean')

    # E: Combine
    total_loss = ce_loss + lambda_adv * kl_loss

    return total_loss, ce_loss.item(), kl_loss.item()


print('adversarial_kl_loss() defined')
print(f'  lambda_adv = {EDAT_LAMBDA_ADV}')
print('  Loss = CE_weighted(clean) + 0.5 * KL(p_clean || p_adv)')
print('  Uses plain CrossEntropy -- matches 03_advanced_training (no focal loss)')


adversarial_kl_loss() defined
  lambda_adv = 0.5
  Loss = CE_weighted(clean) + 0.5 * KL(p_clean || p_adv)
  Uses plain CrossEntropy -- matches 03_advanced_training (no focal loss)


---
## Task 24a — Small Batch Test

Five sequential checks on the 8-sample mini-batch:

1. Loss values are finite, positive, and KL >= 0
2. `total_loss.backward()` completes without error
3. At least one LoRA parameter has a non-zero gradient
4. Optimizer step completes and no weights become NaN
5. Loss changes after the optimizer step (weights were updated)

**Expected ranges on CPU (no GPU):**
- `ce_loss` : 0.5 -- 2.5
- `kl_loss` : 0.05 -- 0.5
- `total`   : 0.5 -- 3.0

In [20]:
import math

print('=' * 60)
print('TASK 24a -- Small Batch Sanity Check')
print('=' * 60)

model.train()  # must be train mode: dropout on, grad tracking on

test_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
)

# CHECK 1: Forward pass + loss values
print('\n[1] Running adversarial_kl_loss on 8-sample batch...')
test_optimizer.zero_grad()

total_loss, ce_val, kl_val = adversarial_kl_loss(
    model, INPUT_IDS, ATTN_MASK, CWE_LABELS,
)

print(f'   ce_loss    = {ce_val:.4f}')
print(f'   kl_loss    = {kl_val:.4f}')
print(f'   total_loss = {total_loss.item():.4f}')

assert not math.isnan(total_loss.item()), 'ERROR: NaN loss -- check labels are in [0,7]'
assert not math.isinf(total_loss.item()), 'ERROR: Inf loss -- dtype or scale issue'
assert total_loss.item() > 0,             'ERROR: loss <= 0 -- something is wrong'
assert kl_val >= 0, 'ERROR: KL < 0 -- check kl_div arg order (log_softmax first)'
print('   PASS: loss is finite and positive')

# CHECK 2: Backward pass
print('\n[2] total_loss.backward()...')
total_loss.backward()
print('   PASS: backward completed')

# CHECK 3: Gradient flow
print('\n[3] Checking gradients on trainable parameters...')
with_grad, zero_grad = [], []
for name, param in model.named_parameters():
    if param.requires_grad:
        if param.grad is not None and param.grad.norm().item() > 0:
            with_grad.append((name, param.grad.norm().item()))
        else:
            zero_grad.append(name)

print(f'   Params with non-zero grad : {len(with_grad)}')
print(f'   Params with zero grad     : {len(zero_grad)}')
for name, norm in with_grad[:3]:
    print(f'     {name[:55]:55s}  grad_norm={norm:.6f}')

assert len(with_grad) > 0, 'ERROR: no gradients -- check nothing is wrongly detached'
print('   PASS: gradients flowing correctly')

# CHECK 4: Optimizer step
print('\n[4] optimizer.step()...')
torch.nn.utils.clip_grad_norm_(
    filter(lambda p: p.requires_grad, model.parameters()),
    max_norm=cfg['max_grad_norm'],
)
test_optimizer.step()

nan_params = [n for n, p in model.named_parameters()
              if p.requires_grad and torch.isnan(p).any()]
assert len(nan_params) == 0, f'ERROR: NaN weights after step: {nan_params[:3]}'
print('   PASS: no NaN weights after optimizer step')

# CHECK 5: Weights updated
print('\n[5] Verifying model weights changed...')
model.train()
with torch.no_grad():
    out2  = model(INPUT_IDS, ATTN_MASK, cwe_labels=CWE_LABELS)
    loss2 = out2['loss'].item()

print(f'   CE loss before step : {ce_val:.4f}')
print(f'   CE loss after step  : {loss2:.4f}')
changed = abs(loss2 - ce_val) > 1e-6
assert changed, 'ERROR: loss unchanged -- model weights were not updated'
print(f'   PASS: weights updated (delta = {abs(loss2 - ce_val):.6f})')

print('\n' + '=' * 60)
print('ALL 5 CHECKS PASSED')
print('Tasks 22a, 23a, 24a complete.')
print('=' * 60)
print()
print('Next: task 26a -- integrate adversarial_kl_loss() into the full training loop.')
print('Add the following to each training step in 03_advanced_training or a new notebook:')
print('  loss = loss_rdrop + scl_weight * loss_scl + lambda_adv * kl_loss_term')


TASK 24a -- Small Batch Sanity Check

[1] Running adversarial_kl_loss on 8-sample batch...
   ce_loss    = 0.0183
   kl_loss    = 0.0689
   total_loss = 0.0527
   PASS: loss is finite and positive

[2] total_loss.backward()...
   PASS: backward completed

[3] Checking gradients on trainable parameters...
   Params with non-zero grad : 78
   Params with zero grad     : 0
     encoder.base_model.model.encoder.layer.0.attention.self  grad_norm=1.651454
     encoder.base_model.model.encoder.layer.0.attention.self  grad_norm=0.328988
     encoder.base_model.model.encoder.layer.0.attention.self  grad_norm=0.083318
   PASS: gradients flowing correctly

[4] optimizer.step()...
   PASS: no NaN weights after optimizer step

[5] Verifying model weights changed...
   CE loss before step : 0.0183
   CE loss after step  : 0.0017
   PASS: weights updated (delta = 0.016546)

ALL 5 CHECKS PASSED
Tasks 22a, 23a, 24a complete.

Next: task 26a -- integrate adversarial_kl_loss() into the full training loop

In [21]:
from torch.utils.data import DataLoader
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import numpy as np

EVAL_BATCH_SIZE = 16

val_ds     = VulnerabilityDataset(cache, split='val')
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
print(f'Validation samples : {len(val_ds)}')


def evaluate(model, loader, mode='clean', epsilon=EDAT_EPSILON,
             alpha=EDAT_ALPHA, num_steps=EDAT_NUM_STEPS):
    """
    mode='clean'       -- standard forward pass under no_grad
    mode='adversarial' -- PGD attack (needs grad), then eval forward under no_grad

    KEY FIX: PGD must run OUTSIDE torch.no_grad() because it calls
    loss.backward() internally to compute gradients w.r.t. delta.
    Only the final clean/adversarial inference step uses no_grad.
    """
    assert mode in ('clean', 'adversarial')
    all_preds, all_labels = [], []

    for batch in loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['cwe_label'].to(DEVICE)

        if mode == 'clean':
            # Clean eval: no gradients needed at all
            model.eval()
            with torch.no_grad():
                logits = model(ids, mask)['logits']

        else:
            # Adversarial eval:
            # Step 1 -- PGD runs with gradients (train mode + no no_grad wrapper)
            model.train()                     # dropout on, grad tracking on
            perturbed = pgd_attack(           # calls backward() internally
                model, ids, mask, labels,
                epsilon=epsilon, alpha=alpha, num_steps=num_steps,
            )                                 # returns detached tensor

            # Step 2 -- final inference on perturbed embeddings (no grad needed)
            model.eval()
            with torch.no_grad():
                logits = model.forward_embeds(perturbed, mask)['logits']

        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

    return np.concatenate(all_preds), np.concatenate(all_labels)


def print_report(preds, labels, title):
    macro_f1 = f1_score(labels, preds, average='macro',  zero_division=0)
    macro_p  = precision_score(labels, preds, average='macro', zero_division=0)
    macro_r  = recall_score(labels, preds, average='macro',  zero_division=0)

    print(f'\n{"=" * 60}')
    print(f'{title}')
    print(f'{"=" * 60}')
    print(f'  Macro F1        : {macro_f1:.4f}')
    print(f'  Macro Precision : {macro_p:.4f}')
    print(f'  Macro Recall    : {macro_r:.4f}')
    print()
    print('Per-class breakdown:')
    print(classification_report(
        labels, preds,
        target_names=CWE_8_CLASSES,
        zero_division=0, digits=3,
    ))
    cm = confusion_matrix(labels, preds)
    print('Confusion matrix (rows=true, cols=predicted):')
    header = ''.join(f'{c:>10}' for c in CWE_8_CLASSES)
    print(f'{"":>12}{header}')
    for i, row in enumerate(cm):
        row_str = ''.join(f'{v:>10}' for v in row)
        print(f'{CWE_8_CLASSES[i]:>12}{row_str}')
    return macro_f1


# CLEAN
print('Running clean evaluation...')
clean_preds, val_labels = evaluate(model, val_loader, mode='clean')
clean_f1 = print_report(clean_preds, val_labels, 'CLEAN EVALUATION')

# ADVERSARIAL
print('\nRunning adversarial evaluation (PGD)...')
print('(Each batch runs 3 PGD steps -- slower than clean eval)')
adv_preds, _ = evaluate(model, val_loader, mode='adversarial')
adv_f1 = print_report(adv_preds, val_labels, 'ADVERSARIAL EVALUATION')

# ROBUSTNESS SUMMARY
print('\n' + '=' * 60)
print('ROBUSTNESS SUMMARY')
print('=' * 60)
print(f'  Clean F1       : {clean_f1:.4f}')
print(f'  Adversarial F1 : {adv_f1:.4f}')
drop = clean_f1 - adv_f1
pct  = drop / clean_f1 * 100 if clean_f1 > 0 else 0
print(f'  F1 drop        : {drop:.4f}  ({pct:.1f}% relative drop)')
print()
if drop < 0.05:
    print('  ROBUST -- small F1 drop under adversarial perturbation')
elif drop < 0.15:
    print('  MODERATELY ROBUST -- consider lower epsilon or more EDAT epochs')
else:
    print('  BRITTLE -- large F1 drop, epsilon is too high')
    print('  Action: reduce EDAT_EPSILON and re-run sensitivity check')


Validation samples : 1528
Running clean evaluation...

CLEAN EVALUATION
  Macro F1        : 0.6296
  Macro Precision : 0.6245
  Macro Recall    : 0.6549

Per-class breakdown:
              precision    recall  f1-score   support

     CWE-077      0.527     0.569     0.547       153
     CWE-601      0.451     0.645     0.531        93
     CWE-022      0.497     0.806     0.615       103
     CWE-094      0.460     0.468     0.464       124
     CWE-089      0.834     0.806     0.820       350
     CWE-352      0.763     0.815     0.788       146
     CWE-079      0.569     0.382     0.457       152
     unknown      0.894     0.749     0.816       407

    accuracy                          0.688      1528
   macro avg      0.624     0.655     0.630      1528
weighted avg      0.710     0.688     0.692      1528

Confusion matrix (rows=true, cols=predicted):
               CWE-077   CWE-601   CWE-022   CWE-094   CWE-089   CWE-352   CWE-079   unknown
     CWE-077        87        13   

In [13]:
# Epsilon sensitivity check
print(f"{'epsilon':>10}  {'ce':>8}  {'kl':>8}  {'total':>8}")
print("-" * 42)
for eps in [0.001, 0.003, 0.005, 0.01, 0.02]:
    model.train()
    _, ce, kl = adversarial_kl_loss(
        model, INPUT_IDS, ATTN_MASK, CWE_LABELS,
        epsilon=eps, alpha=eps/2, num_steps=3
    )
    total = ce + EDAT_LAMBDA_ADV * kl
    flag = " <- good" if 0.05 <= kl <= 1.0 else ""
    print(f"{eps:>10.3f}  {ce:>8.4f}  {kl:>8.4f}  {total:>8.4f}{flag}")

   epsilon        ce        kl     total
------------------------------------------
     0.001    0.0430    0.2508    0.1684 <- good
     0.003    0.0594    1.3508    0.7348
     0.005    0.1124    4.1827    2.2038
     0.010    0.1235   11.6577    5.9523
     0.020    0.0271   17.1821    8.6182


---
## Appendix -- Debugging Guide

**NaN loss:**
- Confirm all `cwe_labels` are in `[0, 7]` (no -1 values)
- Reduce `EDAT_EPSILON` to `0.001` and retry
- Make sure `model.train()` was called (dropout must be active)

**KL loss is negative:**
- You passed `softmax` instead of `log_softmax` as first arg to `F.kl_div`
- Fix: use `F.log_softmax(adv_logits, dim=-1)` as the `input` argument

**Zero gradients (check 3 fails):**
- Check `model.forward_embeds()` passes `inputs_embeds=` as keyword arg
- If PEFT raises an error on `inputs_embeds`, try: `self.encoder.base_model.model(inputs_embeds=..., attention_mask=...)`

**Shape mismatch during PGD:**
- The embedding lookup `embed_fn(input_ids)` gives `(B, L, 768)`
- `delta` must be same shape -- verify with `print(embeds_init.shape)`

**Checkpoint key error:**
- `03_advanced_training` saves key `model_state`
- `02_training` saves key `model_state_dict`
- The loader above tries both automatically

**Expected output on CPU (no GPU):**
```
ce_loss    = 0.8 -- 2.5
kl_loss    = 0.05 -- 0.5
total_loss = 0.8 -- 3.0
```